# dim_channel_scd1 (SCD 1)

One row per `channel_id`, latest values. `channel_key = xxhash64(channel_id)` (same key as `dim_channel`).

Task `dim_channel_scd1` of the job "2. gold (poc tables)"; widgets `load_type` (incremental / full), `from_date`, `to_date` (Vietnam days, empty = yesterday .. tomorrow). The live dim is unchanged.

In [ ]:
%run ./_scd

# Set up environment

In [ ]:
for layer in ("silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
dbutils.widgets.dropdown("load_type", "incremental", ["incremental", "full"])
dbutils.widgets.text("from_date", "")   # yyyy-MM-dd, Vietnam day, inclusive; empty = yesterday
dbutils.widgets.text("to_date", "")     # exclusive; empty = tomorrow

silver = dbutils.widgets.get("silver")
gold = dbutils.widgets.get("gold")
load_type = dbutils.widgets.get("load_type")
from_date, to_date = get_incremental_dates(dbutils.widgets.get("from_date"), dbutils.widgets.get("to_date"))

# First load of the table = full (poc update_load_type: full once, then incremental)
if not spark.catalog.tableExists(f"{gold}.dim_channel_scd1"):
    load_type = "full"
print(f"load_type = {load_type}, window = [{from_date}, {to_date})")

# Read Source Table

`stg_channels` keeps every read (grain `channel_id` × `snapshot_ts`); `stg_channels_latest` = the latest read per channel.

In [ ]:
stg_channels = spark.table(f"{silver}.stg_channels")
stg_channels_latest = spark.table(f"{silver}.stg_channels_latest")

## Incremental: channels read in the window

In [ ]:
if load_type == "incremental":
    changed = (stg_channels
               .filter((F.col("snapshot_date") >= from_date) & (F.col("snapshot_date") < to_date))
               .select("channel_id").distinct())
    stg_channels_latest = stg_channels_latest.join(changed, "channel_id", "left_semi")

# Transformation

In [ ]:
result_df = stg_channels_latest.select(
    F.xxhash64("channel_id").alias("channel_key"),
    F.col("channel_id").cast("string").alias("channel_id"),
    F.col("channel_title").cast("string").alias("channel_title"),
    F.col("channel_description").cast("string").alias("channel_description"),
    F.col("custom_url").cast("string").alias("custom_url"),
    F.col("country").cast("string").alias("country"),
    F.col("default_language").cast("string").alias("default_language"),
    F.col("thumbnail_url").cast("string").alias("thumbnail_url"),
    F.col("published_at").cast("timestamp").alias("published_at"),
    F.col("created_at").cast("timestamp").alias("created_at"),
    F.col("snapshot_ts").cast("timestamp").alias("modified_at"),   # when these values were read
)

## Add metadata

In [ ]:
insert_date = datetime.now()
etl_by = "Databricks"
result_df = result_df.withColumn("etl_date", F.lit(insert_date)).withColumn("etl_by", F.lit(etl_by))

## Add checksums

`created_at` / `modified_at` are not hashed: a re-read with the same values writes nothing.

In [ ]:
primary_key = ["channel_id"]
result_df = add_checksums(result_df, primary_key, exclude=["channel_key", "created_at", "modified_at"])

## Create table

In [ ]:
path = f"{gold}.dim_channel_scd1"
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {path} (
        channel_key BIGINT, channel_id STRING, channel_title STRING, channel_description STRING, custom_url STRING,
        country STRING, default_language STRING, thumbnail_url STRING, published_at TIMESTAMP,
        created_at TIMESTAMP, modified_at TIMESTAMP, etl_date TIMESTAMP, etl_by STRING,
        primary_checksum STRING, record_checksum STRING
    ) USING DELTA
    COMMENT 'Channel dimension, SCD 1 (poc checksum method). channel_key = xxhash64(channel_id), -1 = Unknown.'
""")
result_df = result_df.select(*spark.table(path).columns)

# Write to Table

**full**: overwrite (poc `full_refresh`). **incremental**: MERGE on `primary_checksum`; update only when `record_checksum` changed and the source row is not older (`s.modified_at >= t.modified_at`, so a backfill of an old day never overwrites newer values); insert new keys.

In [ ]:
if load_type == "full":
    full_refresh(result_df, path)
else:
    result_df.createOrReplaceTempView("source")
    spark.sql(f"""
        MERGE INTO {path} AS t
        USING source AS s
        ON t.primary_checksum = s.primary_checksum
        WHEN MATCHED AND t.record_checksum <> s.record_checksum
             AND (t.modified_at IS NULL OR s.modified_at >= t.modified_at) THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

## Unknown row

Key −1, natural key NULL: facts with no match point to it. Inserted only when missing (no empty commit).

In [ ]:
if spark.table(path).where("channel_key = -1").isEmpty():
    spark.sql(f"INSERT INTO {path} (channel_key, channel_title, etl_date, etl_by) VALUES (-1, 'Unknown', current_timestamp(), 'Databricks')")

## Check

In [ ]:
dup = spark.sql(f"SELECT COUNT(*) - COUNT(DISTINCT channel_id) FROM {path} WHERE channel_id IS NOT NULL").first()[0]
assert dup == 0, f"{path}: {dup} duplicate channel_id"
last = spark.sql(f"DESCRIBE HISTORY {path} LIMIT 2").where("operation <> 'WRITE'").select("operation", "operationMetrics").first()
print(f"✅ {path}: {spark.table(path).count()} rows", "·", last["operation"], dict(last["operationMetrics"] or {}) if last else "")